In [ ]:
from cfspopcon.formulas.energy_confinement.read_energy_confinement_scalings import ConfinementScaling, read_confinement_scalings
read_confinement_scalings()

energy_confinement_scaling = 'ITER98y2'
scaling = ConfinementScaling.instances[energy_confinement_scaling]

print(dir(scaling))


In [ ]:
%load_ext autoreload
%autoreload 2

from pprint import pprint

from popsim.simulate import CombinatorialCases, MultiCases, make_time_base
from popsim.simulators.comet_mirror.scenarios.sparc_prd import build_comet_mirror_config

# Initialize the simulator.
model, state, params = build_comet_mirror_config()

# Make a time base for all of our simulations.
time_base = make_time_base(t0=0.0, t1=5.0, dt=0.01)

pprint(state)
pprint(params)

In [ ]:
from copy import deepcopy

import jax.numpy as jnp

from popsim.enums import Impurity
from popsim.simulate import SimInput, simulate

# Create a copy of "params". It's best to keep the original one around untouched.
new_params = deepcopy(params)

# Manually define a current-ramp where the key is the time in seconds and the value is the current in Amperes.
new_params.plasma_current = {0.0: 8.7e6, 1.0: 8.7e6, 5.0: 4.0e6}

# Manually define an auxiliary heating power ramp where the key is the time in seconds and the value is the power in MW.
new_params.P_aux_MW = {0.0: 11.1, 5.0: 7.0}

# Manually define a quick tungsten spike. Note that under the hood linear interpolation is happening, so we need this
# perhaps somewhat awkward definition.
new_params.fueling19[Impurity.Tungsten] = {
    0.0: 0.0,
    1.99: 0.0,  # Start ramping impurities.
    2.0: 0.1,  # Impurity injection.
    2.1: 0.1,  # Impurity injection holding.
    2.11: 0.0,  # Impurity drops back to 0.0.
    5.0: 0.0,  # Impurity holds at 0.0.
}


# Build the simulation inputs.
sim_inputs = SimInput(time=time_base, initial_state=state, params=new_params)

dataset = simulate(
    module=model,
    sim_inputs=sim_inputs,
)

In [ ]:
dataset

In [ ]:
import holoviews as hv

from popsim.visualize import visualize_time_series

hv.extension("matplotlib")

visualize_vars = [
    "output.aux_data.params.plasma_current",
    "output.aux_data.params.P_aux_MW",
    "output.aux_data.params.fueling19.Impurity.Tungsten",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.hmode_state.hmode",
    "output.aux_data.Prad_imp_MW",
    "output.aux_data.P_rad_MW",
    "output.aux_data.tau_E",
    "output.aux_data.average_electron_density_19",
]
visualize_time_series(dataset[visualize_vars])